<a href="https://colab.research.google.com/github/areebazia-lsh/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal checks and baseline rule

### Signal 1: Staleness

I will treat a page as stale when it has not been updated for at least 180 days. I will check how page exposure varies across staleness buckets before using this signal in the rule.

### Signal 2: CTR versus position

I will check how CTR changes across average-position buckets. This is linked to the CTR-fix logic because pages that are visible but receive relatively few clicks may be worth review.

The baseline will use only signals known at the decision moment. It will not use trend_direction, trend_pct, or any future-window outcome.

In [21]:
import os

print("Current directory:", os.getcwd())
print("Content folders:", os.listdir("/content"))

Current directory: /content/flyrank-ml-internship
Content folders: ['.config', 'flyrank-ml-internship', 'sample_data']


In [22]:
!git clone https://github.com/areebazia-lsh/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [23]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [24]:
!ls data/raw

content_refresh_anonymized.csv


In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Loaded successfully!
Rows: 30000
Columns: 44


In [26]:
import pandas as pd
import numpy as np
import os

paths = [
    "data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv"
]

df = None

for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Loaded:", path)
        break

if df is None:
    raise FileNotFoundError("Starter CSV not found.")

Loaded: data/raw/content_refresh_anonymized.csv


In [27]:
# Signal 1: Staleness
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 89, 179, 359, np.inf],
    labels=["0-89 days", "90-179 days", "180-359 days", "360+ days"]
)

stale_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          mean_impressions_90d=("impressions_90d", "mean"),
          median_impressions_90d=("impressions_90d", "median")
      )
      .reset_index()
)

print(stale_check)

  staleness_bucket      n  mean_impressions_90d  median_impressions_90d
0        0-89 days  20655           4219.161317                   472.0
1      90-179 days   9171           7486.665140                  1692.0
2     180-359 days    169           1206.893491                    16.0
3        360+ days      5              8.200000                     2.0


In [28]:
# Signal 2: CTR versus position
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-np.inf, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"]
)

position_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          mean_ctr=("ctr", "mean"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print(position_check)

  position_bucket      n  mean_ctr  median_ctr
0             1-3   2346  1.472869        0.00
1            4-10  11842  0.651045        0.16
2           11-20   7273  0.323443        0.10
3             21+   8539  0.211333        0.00


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [29]:
# Build the baseline score

df["is_stale"] = (
    df["days_since_last_update"] >= 180
).astype(int)

df["is_visible"] = (
    df["impressions_90d"] >= 500
).astype(int)

df["is_ctr_opportunity"] = (
    df["avg_position"].between(1, 20, inclusive="both")
    & (df["ctr"] < 0.30)
).astype(int)

df["baseline_score"] = (
    2 * df["is_stale"]
    + 2 * df["is_ctr_opportunity"]
    + 1 * df["is_visible"]
)

conditions = [
    (df["is_stale"] == 1) & (df["is_visible"] == 1),
    df["is_ctr_opportunity"] == 1,
    df["is_stale"] == 1
]

choices = [
    "STALE_VISIBLE",
    "CTR_OPPORTUNITY",
    "STALE"
]

df["reason_code"] = np.select(
    conditions,
    choices,
    default="MONITOR"
)

action_map = {
    "STALE_VISIBLE": "Refresh content",
    "CTR_OPPORTUNITY": "Review CTR opportunity",
    "STALE": "Review for refresh",
    "MONITOR": "Monitor"
}

df["action"] = df["reason_code"].map(action_map)

queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

output = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_type"
    ]
].copy()

os.makedirs("work/outputs", exist_ok=True)

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows ranked:", len(output))
print("Saved: work/outputs/baseline_action_score.csv")
print("\nTop 10:")
print(output.head(10).to_string(index=False))

Rows ranked: 30000
Saved: work/outputs/baseline_action_score.csv

Top 10:
 rank           content_id         client_id  baseline_score     reason_code                 action  days_since_last_update  impressions_90d  avg_position  ctr    content_type
    1 content_cf56e2e2e282 client_7f2253d7e2               5   STALE_VISIBLE        Refresh content                     194            61678          19.7 0.15 keyword article
    2 content_c2d929d83eaa client_7f2253d7e2               5   STALE_VISIBLE        Refresh content                     193             7558          17.9 0.20 keyword article
    3 content_928af3e22c80 client_7f2253d7e2               5   STALE_VISIBLE        Refresh content                     193             1697          15.8 0.12 keyword article
    4 content_e3ff1b093148 client_d029fa3a95               5   STALE_VISIBLE        Refresh content                     183             1408           7.8 0.28 keyword article
    5 content_77d4d5930e5e client_7f2253d7e2  

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [30]:
top20 = output.head(20).copy()

top20["confidence_note"] = np.where(
    top20["baseline_score"] >= 4,
    "Higher-priority because multiple baseline signals agree.",
    "Lower-confidence because fewer signals support the recommendation."
)

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"].eq("STALE_VISIBLE"),
        top20["reason_code"].eq("CTR_OPPORTUNITY"),
        top20["reason_code"].eq("STALE")
    ],
    [
        "The page may already be performing well despite being old.",
        "Low CTR may be normal for its position or search intent.",
        "The page may be stale without actually needing a refresh."
    ],
    default="The page may not need action despite weak baseline signals."
)

print(
    top20[
        [
            "rank",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ].to_string(index=False)
)

 rank                 action     reason_code                                          confidence_note                                   what_would_make_it_wrong
    1        Refresh content   STALE_VISIBLE Higher-priority because multiple baseline signals agree. The page may already be performing well despite being old.
    2        Refresh content   STALE_VISIBLE Higher-priority because multiple baseline signals agree. The page may already be performing well despite being old.
    3        Refresh content   STALE_VISIBLE Higher-priority because multiple baseline signals agree. The page may already be performing well despite being old.
    4        Refresh content   STALE_VISIBLE Higher-priority because multiple baseline signals agree. The page may already be performing well despite being old.
    5        Refresh content   STALE_VISIBLE Higher-priority because multiple baseline signals agree. The page may already be performing well despite being old.
    6        Refresh content   STA

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Signal verdicts

- **Staleness: CONFIRMED** — In the starter data, the 180+ day buckets have much lower median impressions than the newer buckets, so staleness shows a clear directional relationship with exposure.
- **CTR versus position: CONFIRMED** — Mean CTR decreases across the position buckets, from 1.473 for positions 1–3 to 0.211 for positions 21+, supporting this as a useful directional signal.

The baseline does not use `trend_direction` or `trend_pct` because these fields are outcome-derived and would create leakage.

- **Staleness: CONFIRMED** — In the starter data, the 180+ day buckets have much lower median impressions than the newer buckets, so staleness shows a clear directional relationship with exposure.
- **CTR versus position: CONFIRMED** — Mean CTR decreases across the position buckets, from 1.473 for positions 1–3 to 0.211 for 21+, supporting this as a useful directional signal.

In [31]:
baseline_inputs = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

excluded_leakage_fields = [
    "trend_direction",
    "trend_pct"
]

print("Baseline inputs:", baseline_inputs)
print("Excluded leakage fields:", excluded_leakage_fields)

print("\nLeakage check:")
print("trend_direction used?", "trend_direction" in baseline_inputs)
print("trend_pct used?", "trend_pct" in baseline_inputs)

Baseline inputs: ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
Excluded leakage fields: ['trend_direction', 'trend_pct']

Leakage check:
trend_direction used? False
trend_pct used? False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.